In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, gc
gc.collect()
torch.cuda.empty_cache()

print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")
print(f"Total VRAM: {torch.cuda.mem_get_info()[1] / 1024**3:.2f} GB")

Free VRAM: 10.92 GB
Total VRAM: 14.56 GB


In [ ]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

## Load Dataset

In [ ]:
train = pd.read_csv("samsum-train.csv")
val = pd.read_csv("samsum-validation.csv")

In [ ]:
train.sample()

,id,dialogue,summary
4742,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.


## Preprocess

In [ ]:
train.dropna(inplace=True)

In [ ]:
train = train.sample(n=10000, random_state=42).reset_index(drop=True)

In [ ]:
import re

In [ ]:
def clean_data(text):
    #remove next lines
    text = re.sub(r"\r\n", " ", text)
    #remove html tages
    text = re.sub(r"<.*?>", " ", text)
    #remove extra spaces
    text = re.sub(r"\s+", " ", text)
    #remove trailing spaces and convert text into lowercase
    text = text.strip().lower()

    return text

In [ ]:
train["dialogue"] = train["dialogue"].apply(clean_data)
train["summary"] = train["summary"].apply(clean_data)

val["dialogue"] = val["dialogue"].apply(clean_data)
val["summary"] = val["summary"].apply(clean_data)

## Tokenize

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-base")

In [ ]:
def tokenize(data):
    inputs = tokenizer(
        data["dialogue"],
        padding="max_length",
        max_length=512,
        truncation=True
    )

    target = tokenizer(
        data["summary"],
        padding="max_length",
        max_length=150,
        turncation=True
    )

    inputs["labels"] = target["input_ids"]

    return inputs

In [ ]:
training_dataset = train.apply(tokenize, axis=1).tolist()
validation_dataset = val.apply(tokenize, axis=1).tolist()

In [ ]:
training_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

# Fine Tune T5 Tokenizer

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-base")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")
model.to(device)

Device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [ ]:
training_arg = TrainingArguments(
    output_dir = "./results",
    num_train_epochs= 6,
    weight_decay = 0.01,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size = 4,
    gradient_accumulation_steps=2,
    fp16=True,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    warmup_steps = 500
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_arg,
    train_dataset = training_dataset,
    eval_dataset = validation_dataset,
)

In [ ]:
# Train model
trainer.train()

Epoch,Training Loss,Validation Loss


In [ ]:
model.save_pretrained("./saved_summarizer_model")
tokenizer.save_pretrained("./saved_summarizer_model")

# Test model's Performance

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summarizer_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summarizer_model")

In [ ]:
def summarize_dialogue(dialogue):
  #clean data
  dialogue = clean_data(dialogue)

  #tokenize dialogue
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  ).to(device)

  #generate summary (token ids)
  model.to(device)
  target = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length = 150,
      num_beams = 4,
      early_stopping = True
  )

  #decode (token ids -> summary)
  summary = tokenizer.decode(
      target[0],
      skip_special_tokens = True
  )

  return summary

In [ ]:
#Check final results:

text = """
Artificial Intelligence has rapidly transformed the technology industry over the past decade. Earlier, most software applications were designed using fixed rules written manually by programmers. However, modern AI systems are capable of learning patterns directly from data, allowing them to make predictions, generate content, and automate complex tasks. Machine Learning, which is a subset of AI, enables computers to improve their performance without being explicitly programmed for every scenario.

One of the biggest breakthroughs in recent years has been the development of Deep Learning models. These models use neural networks with multiple layers to process large amounts of data. Deep Learning has achieved impressive results in areas such as image recognition, speech processing, recommendation systems, and natural language understanding. Applications like voice assistants, chatbots, self-driving cars, and medical diagnosis systems rely heavily on these technologies.

Natural Language Processing, commonly known as NLP, is another important branch of AI. NLP focuses on enabling machines to understand and generate human language. Earlier NLP systems struggled to understand context and meaning, but modern transformer-based architectures have significantly improved language understanding. Models such as GPT and BERT can perform tasks like translation, summarization, question answering, and text generation with high accuracy.

Despite these advancements, AI also introduces several challenges. Ethical concerns such as bias in datasets, privacy issues, misinformation, and job displacement are becoming increasingly important. Researchers and organizations are working to build responsible AI systems that are transparent, fair, and safe for society. Governments around the world are also discussing regulations to ensure that AI technologies are developed and used responsibly.

In the future, AI is expected to become even more integrated into daily life. From personalized education systems to intelligent healthcare assistants and advanced robotics, AI has the potential to improve efficiency and solve complex global problems. However, balancing innovation with ethical responsibility will remain one of the most critical aspects of AI development.
"""
summary = summarize_dialogue(text)
print(summary)

In [ ]:
!zip -r model.zip saved_summarizer_model

from google.colab import files
files.download("model.zip")